In [55]:
from tavily import TavilyClient
from datetime import datetime, timedelta
import json
import asyncio
import json
from typing import List, Dict, Any, Optional

from pydantic import BaseModel, Field, ValidationError
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Import shared clients
import shared_clients
from shared_clients import shared_clients


#!/usr/bin/env python3
"""
Manager Multiprocessing - Import Manager Agent and run multiprocessing analysis
"""

import asyncio
import multiprocessing as mp
from typing import List, Dict, Any
from concurrent.futures import ProcessPoolExecutor
from time import time


#!/usr/bin/env python3
"""
Simple Multiprocessing Manager - Direct multiprocessing without classes
"""

import asyncio
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from Manager_Agent import quick_analysis

import asyncio
import json
from typing import Dict, Any

# Import shared clients
import shared_clients
from shared_clients import shared_clients



# Section 1). Search in Internet, what is the current long/short logic 

In [56]:
from tavily import TavilyClient

def get_stock_analysis_tavily_dual(ticker):
    """
    Get stock analysis using two Tavily searches and integrate results
    
    Args:
        ticker (str): Stock ticker symbol (e.g., 'FICO', 'NVDA')
    
    Returns:
        dict: Contains summary['sell_and_buy'] (as list) and summary['business_logic']
    """
    try:
        # Initialize Tavily client
        client = TavilyClient("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")
        
        print(f"🔍 Starting dual Tavily search for {ticker}...")
        
        # SEARCH 1: Sell/Buy Analysis (Market focus, validation, fundamental level)
        query1 = f"List 3–9 {ticker} drivers (Macro/Market/Fundamentals), with +/– factors tied to future growth, risks, or resolutions. Return ONLY numbered list(each line summarize as key words bullet point)."
        print(f"📊 Search 1: Sell/Buy analysis for {ticker}...")
        
        response1 = client.search(
            query=query1,
            include_answer="advanced",
            search_depth="advanced",
            topic="general",
            max_results=10
        )
        
        # SEARCH 2: Business Logic Analysis (Moat, business model)
        query2 = f"I want a very specific deep detailed explanation on the business model: moat, scale, customers, and logic for the stock ticker \"{ticker}\", if no, just say some postive and negative about the company"
        
        print(f"�� Search 2: Business logic analysis for {ticker}...")
        
        response2 = client.search(
            query=query2,
            include_answer="advanced",
            search_depth="advanced",
            topic="general",
            max_results=10
        )
        
        # Extract content and answer from Search 1 (Sell/Buy)
        sell_buy_content_list = []
        if 'results' in response1:
            for result in response1['results']:
                if 'content' in result and result['content']:
                    sell_buy_content_list.append(result['content'])
        
        sell_buy_content = " | ".join(sell_buy_content_list)
        sell_buy_answer = response1.get('answer', '')
        
        # Extract content and answer from Search 2 (Business Logic)
        business_logic_content_list = []
        if 'results' in response2:
            for result in response2['results']:
                if 'content' in result and result['content']:
                    business_logic_content_list.append(result['content'])
        
        business_logic_content = " | ".join(business_logic_content_list)
        business_logic_answer = response2.get('answer', '')
        
        # Create structured summary
        summary = {
            'sell_and_buy': sell_buy_answer,  # This will now be a numbered list
            'business_logic': business_logic_answer
        }
        
        # Create result dictionary
        result = {
            'ticker': ticker,
            'summary': summary,
            'sell_and_buy_content': sell_buy_content,
            'business_logic_content': business_logic_content,
            'sell_and_buy_response_time': response1.get('response_time', 0),
            'business_logic_response_time': response2.get('response_time', 0),
            'sell_and_buy_results': len(response1.get('results', [])),
            'business_logic_results': len(response2.get('results', []))
        }
        
        print(f"✅ Dual Tavily search complete for {ticker}")
        print(f"💰 Sell/Buy results: {result['sell_and_buy_results']}")
        print(f"🏢 Business Logic results: {result['business_logic_results']}")
        print(f"⏱️ Total response time: {result['sell_and_buy_response_time'] + result['business_logic_response_time']} seconds")
        
        return result
        
    except Exception as e:
        print(f"❌ Error in dual Tavily search for {ticker}: {e}")
        return {
            'ticker': ticker,
            'summary': {
                'sell_and_buy': 'Error occurred during sell/buy search',
                'business_logic': 'Error occurred during business logic search'
            },
            'sell_and_buy_content': '',
            'business_logic_content': ''
        }

# Simple usage function
def get_stock_summary_simple(ticker):
    """
    Simple function to get stock summary - just input ticker
    
    Args:
        ticker (str): Stock ticker symbol
    
    Returns:
        dict: Summary with sell_and_buy (as list) and business_logic
    """
    result = get_stock_analysis_tavily_dual(ticker)
    return result['summary']

In [58]:
ticker =  "PLTR"
summary = get_stock_summary_simple(ticker)
sell_and_buy = summary['sell_and_buy']
business_logic = summary['business_logic']
language = "Chinese"

🔍 Starting dual Tavily search for PLTR...
📊 Search 1: Sell/Buy analysis for PLTR...
�� Search 2: Business logic analysis for PLTR...
✅ Dual Tavily search complete for PLTR
💰 Sell/Buy results: 10
🏢 Business Logic results: 10
⏱️ Total response time: 24.68 seconds


## In future, Frontend need to print these two things out

In [59]:
print(f"Sell and Buy: {sell_and_buy}")
print(f"Business Logic: {business_logic}")

Sell and Buy: 1. **AI Market Sentiment (+/-)** - Fluctuating enthusiasm for artificial intelligence creates volatility in stock performance and commercial prospects

2. **Valuation Risk (-)** - Trading at extremely high multiples with stock surging 5x, creating bubble risk and vulnerability to sentiment shifts

3. **Economic Growth Sensitivity (-)** - Slower economic growth and potential recession could impact government and enterprise spending on data analytics

4. **Trade Policy Impact (-)** - Protectionist policies may affect international expansion and cross-border data operations

5. **Investor Rotation Risk (-)** - Market shifts away from high-growth stocks during downturns historically impact PLTR performance significantly

6. **Government Contract Dependency (+/-)** - Heavy reliance on government contracts provides stability but limits diversification and growth potential

7. **Commercial Market Expansion (+)** - Growing enterprise adoption of data analytics and AI solutions dr

## Calling supervisor Agent, to generate sub queries for the manager agents

In [60]:
#!/usr/bin/env python3

from __future__ import annotations

from typing import List, Dict, Optional, Literal
from pydantic import BaseModel, Field, ValidationError, conint
from langchain_deepseek import ChatDeepSeek
import os

# Set API key
os.environ["DEEPSEEK_API_KEY"] = "sk-43e9043c7ab8480393d34367f2ae997e"

# -------------------------
# 1) Define schemas (Pydantic v2)
# -------------------------
Priority = conint(ge=1, le=10)  # 1..10
Impact = Literal["Positive", "Negative", "Neutral", "Mixed"]

class SubManagerQuery(BaseModel):
    manager_id: str = Field(..., description="Unique ID (A, B, C, D...)")
    query: str = Field(..., description="Specific query for this sub-manager to investigate")
    dimension: str = Field(..., description="Market dimension this query focuses on")
    priority: Priority = Field(..., description="Priority level 1-10 (1=highest priority)")
    expected_impact: Impact = Field(..., description="Expected impact type")

class SupervisorResult(BaseModel):
    total_sub_managers: conint(ge=1, le=10) = Field(..., description="Number of sub-managers (1-10)")
    sub_managers: List[SubManagerQuery] = Field(default_factory=list, description="List of sub-manager queries")
    buy_side_summary: List[str] = Field(default_factory=list, description="Summary points from buy side")
    sell_side_summary: List[str] = Field(default_factory=list, description="Summary points from sell side")
    business_logic: List[str] = Field(default_factory=list, description="Key business logic insights")
    manager_agent_query: str = Field(..., description="Rephrased and summarized query for Manager Agent")


# -------------------------
# 2) Supervisor Agent - FIXED
# -------------------------
class SupervisorAgent:
    def __init__(self):
        """Initialize the Supervisor Agent (lazy clients)."""
        self.llm: Optional[ChatDeepSeek] = None
        self.structured_llm = None

    def initialize_shared_clients(self):
        """Initialize shared clients synchronously."""
        try:
            from LLM_Call_Agent import DEEPSEEK_API_KEY
            self.llm = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
                api_key=DEEPSEEK_API_KEY
            )
        except ImportError:
            self.llm = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
            )

        self.structured_llm = self.llm.with_structured_output(SupervisorResult)
        print("✅ Supervisor Agent initialized with LangChain DeepSeek structured output")

    def breakdown_query(
        self,
        sell_buy_analysis: str,
        business_logic: str,
        ticker: str,
        language: str
    ) -> SupervisorResult:
        """Break down the query into sub-manager tasks and create Manager Agent query."""
        try:
            if self.structured_llm is None:
                self.initialize_shared_clients()

            print(f"🔍 Supervisor Agent analyzing {ticker} ...")

            # SIMPLIFIED PROMPT - Much cleaner and clearer
            prompt = f"""You are a Supervisor Agent analyzing {ticker} stock.

SELL/BUY ANALYSIS:
{sell_buy_analysis}

BUSINESS LOGIC:
{business_logic}

TASK: Create 1-6 sub-manager queries based on the analysis above.

REQUIREMENTS:
1. Extract key points from sell/buy analysis
2. Create specific sub-queries for investigation
3. Each query should be 30 words or less
4. Output in {language}

OUTPUT FORMAT:
- total_sub_managers: number (1-10)
- sub_managers: list with manager_id, query, dimension, priority (1-10), expected_impact
- buy_side_summary: list of positive points
- sell_side_summary: list of negative points  
- business_logic: list of business insights
- manager_agent_query: rephrased summary for Manager Agent

All text must be in {language}."""

            print("🤖 Making LLM call...")
            result: SupervisorResult = self.structured_llm.invoke(prompt)
            
            # Debug: Check if result is None
            if result is None:
                print("❌ LLM returned None - trying alternative approach...")
                # Try a simpler approach
                simple_prompt = f"Analyze {ticker} stock. Create 3 sub-queries for investigation. Output in {language}."
                result = self.structured_llm.invoke(simple_prompt)
                
                if result is None:
                    raise Exception("LLM returned None even with simple prompt - check API key and connection")
            
            print(f"✅ LLM response received: {type(result)}")
            print(f"✅ Result has total_sub_managers: {hasattr(result, 'total_sub_managers')}")

            if result.total_sub_managers != len(result.sub_managers):
                result.total_sub_managers = len(result.sub_managers)

            print("✅ Supervisor breakdown complete!")
            print(f"→ Total sub-managers: {result.total_sub_managers}")
            print(f"→ Manager Agent Query: {result.manager_agent_query}")
            return result

        except ValidationError as ve:
            print(f"❌ Validation failed: {ve}")
            print(f"   - Error details: {ve}")
            raise Exception(f"Supervisor Agent validation failed: {ve}")

        except Exception as e:
            print(f"❌ Error in supervisor breakdown: {e}")
            print(f"   - Error type: {type(e)}")
            print(f"   - Error details: {str(e)}")
            raise Exception(f"Supervisor Agent processing failed: {e}")

    def get_sub_manager_queries(self, result: SupervisorResult) -> Dict[str, str]:
        """Extract sub-manager queries as a {manager_id: query} dict."""
        return {sm.manager_id: sm.query for sm in result.sub_managers}

    def get_manager_agent_query(self, result: SupervisorResult) -> str:
        """Extract the rephrased Manager Agent query."""
        return result.manager_agent_query

    def print_breakdown_summary(self, result: SupervisorResult) -> None:
        """Pretty print a summary of the breakdown."""
        print("\n" + "=" * 60)
        print("�� SUPERVISOR SUMMARY")
        print("=" * 60)
        print(f"🎯 Sub-Managers: {result.total_sub_managers}")
        print(f"🎯 Manager Agent Query: {result.manager_agent_query}")

        print("\n📈 BUY SIDE SUMMARY:")
        for i, point in enumerate(result.buy_side_summary, 1):
            print(f"  {i}. {point}")

        print("\n📉 SELL SIDE SUMMARY:")
        for i, point in enumerate(result.sell_side_summary, 1):
            print(f"  {i}. {point}")

        print("\n🏢 BUSINESS LOGIC:")
        for i, point in enumerate(result.business_logic, 1):
            print(f"  {i}. {point}")

        print("\n🔍 SUB-MANAGER QUERIES:")
        for sm in result.sub_managers:
            print(f"  {sm.manager_id}. [{sm.priority}] {sm.dimension} — {sm.expected_impact}")
            print(f"     {sm.query}\n")


# -------------------------
# 4) Convenience functions - FIXED
# -------------------------
def create_supervisor_breakdown(
    sell_buy_analysis: str, business_logic: str, ticker: str, language: str
) -> SupervisorResult:
    try:
        sup = SupervisorAgent()
        result = sup.breakdown_query(sell_buy_analysis, business_logic, ticker, language)
        
        # Check if result is None
        if result is None:
            raise Exception("Supervisor Agent returned None - check LLM response")
        
        return result
        
    except Exception as e:
        print(f"❌ Error in create_supervisor_breakdown: {e}")
        print(f"   - sell_buy_analysis length: {len(sell_buy_analysis) if sell_buy_analysis else 0}")
        print(f"   - business_logic length: {len(business_logic) if business_logic else 0}")
        print(f"   - ticker: {ticker}")
        print(f"   - language: {language}")
        raise Exception(f"Failed to create supervisor breakdown: {e}")


def get_sub_manager_queries_simple(result: SupervisorResult) -> Dict[str, str]:
    return {sm.manager_id: sm.query for sm in result.sub_managers}


def get_manager_agent_query_simple(result: SupervisorResult) -> str:
    return result.manager_agent_query

In [61]:

internet_search_result = create_supervisor_breakdown(sell_and_buy, business_logic, ticker, language)

# Access results
print(f"Total sub-managers: {internet_search_result.total_sub_managers}")
for sub_manager in internet_search_result.sub_managers:
    print(f"Manager {sub_manager.manager_id}: {sub_manager.query}")

# Get queries as dictionary
queries = get_sub_manager_queries_simple(internet_search_result)
print(f"Queries: {queries}")

✅ Supervisor Agent initialized with LangChain DeepSeek structured output
🔍 Supervisor Agent analyzing PLTR ...
🤖 Making LLM call...
2025-09-09 11:04:05,802 - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
✅ LLM response received: <class '__main__.SupervisorResult'>
✅ Result has total_sub_managers: True
✅ Supervisor breakdown complete!
→ Total sub-managers: 6
→ Manager Agent Query: 综合分析PLTR股票的投资价值：评估高估值风险与AI驱动增长潜力之间的平衡，考虑政府合同稳定性与商业扩张机会，同时分析竞争环境和市场情绪波动的影响
Total sub-managers: 6
Manager A: 分析PLTR当前估值倍数是否过高，是否存在泡沫风险及回调可能性
Manager B: 评估AI市场情绪波动对PLTR商业前景和股价表现的具体影响机制
Manager C: 研究经济放缓对政府和企业数据分析支出的实际影响程度
Manager D: 分析PLTR商业市场扩张进展和收入多元化战略效果
Manager E: 评估技术突破形态和近期整合后的牛市运行潜力
Manager F: 分析竞争护城河强度及面对科技巨头和AI初创公司竞争的应对能力
Queries: {'A': '分析PLTR当前估值倍数是否过高，是否存在泡沫风险及回调可能性', 'B': '评估AI市场情绪波动对PLTR商业前景和股价表现的具体影响机制', 'C': '研究经济放缓对政府和企业数据分析支出的实际影响程度', 'D': '分析PLTR商业市场扩张进展和收入多元化战略效果', 'E': '评估技术突破形态和近期整合后的牛市运行潜力', 'F': '分析竞争护城河强度及面对科技巨头和AI初创公司竞争的应对能力'}


### Make Sure Frontend Print this out


In [62]:
internet_search_result
print(internet_search_result.sell_side_summary)
print(internet_search_result.buy_side_summary)
print(internet_search_result.business_logic)

sell_side_summary = internet_search_result.sell_side_summary
buy_side_summary = internet_search_result.buy_side_summary
business_logic = internet_search_result.business_logic


['估值倍数极高，存在泡沫风险', '经济放缓可能影响政府和企业支出', '保护主义政策影响国际扩张', '投资者轮动风险在高增长股下跌时显著', '政府合同依赖限制多元化', '高市盈率超过200倍，内部人士大量抛售']
['AI平台驱动80%合同，提供20-30%效率增益', '美国政府收入增长45%，商业收入增长71%', '客户数量增长39%至498家，大额交易显著增加', '净美元留存率128%，运营利润率46%', '技术突破形态显示牛市运行潜力', '商业市场扩张推动收入多元化']
['双核心商业模式：政府与商业市场旗舰平台', '20年发展历史，CIA背景，DISA IL6安全认证', '客户粘性强，切换成本高，账户扩展能力强', '典型交易规模大，前20客户平均超2500万美元', '实施复杂昂贵，限制中小型企业可扩展性', '总可寻址市场限于西方联盟实体']


# Section2). Mutliprocess To Call All Manager  + Chain of Thought AI

In [63]:
# Reload all modules to ensure latest versions
import importlib
import sys
import asyncio
from typing import List, Dict, Any

# Force clear any cached instances
modules_to_clear = [
    'Manager_Agent',
    'shared_clients',
    'Chain_of_Thought_Agent',
    'Market_Expectation_Agent',
    'Revenue_Segmentation_Read_Agent',
    'Macro_Analyst_Agent',
    'Financial_Metrics_Analyst_Agent'
]

for module in modules_to_clear:
    if module in sys.modules:
        del sys.modules[module]

# Reload all modules
modules_to_reload = [
    'Manager_Agent',
    'Market_Expectation_Agent',
    'Revenue_Segmentation_Read_Agent',
    'Macro_Analyst_Agent',
    'Financial_Metrics_Analyst_Agent',
    'Chain_of_Thought_Agent'
]

for module in modules_to_reload:
    importlib.reload(__import__(module))

# Import the functions we need
from Manager_Agent import quick_analysis, get_manager_result, get_manager_progress, get_manager_instance
from Chain_of_Thought_Agent import (
    ChainOfThoughtAgent,
    ChainOfThoughtResult,
    generate_mermaid_code,
    concurrent_call_generate_impact_chain_with_mermaid_auto
)

print("✅ All modules reloaded and imported successfully!")
print("📦 Available functions:")
print("   - quick_analysis")
print("   - get_manager_instance")
print("   - concurrent_call_generate_impact_chain_with_mermaid_auto")



# Convert queries dictionary to list for proper indexing
queries_list = list(queries.values())
query_keys = list(queries.keys())

print(f"📋 Total queries: {len(queries_list)}")
print(f"🔑 Query keys: {query_keys}")
print(f"📝 Query list: {queries_list}")
print(f"🌐 Language setting: {language}")

# CRITICAL: Initialize Manager Agent for concurrent processing
print("🚀 Initializing Manager Agent for concurrent processing...")
manager = await get_manager_instance()

print("✅ Manager Agent initialized and ready for concurrent calls")

async def process_single_query_concurrent(query_index: int, language: str = "English"):
    """Process a single query with multiprocessing - NO user_query needed!"""
    
    try:
        # Get the query from the list
        user_query = queries_list[query_index]
        
        # Use the pre-initialized manager instance directly
        # Manager Agent will use its own shared clients instance
        final_results, agents_result = await manager.run_complete_analysis(
            user_query=user_query,
            ticker=ticker,
            language=language
        )
        
        # Create the same structure as quick_analysis
        manager_result = {
            "user_query": user_query,
            "ticker": ticker,
            "user_id": f"concurrent_{query_index}",
            "agent_results": final_results,
            "agents_result": agents_result,
            "execution_summary": {
                "total_agents_executed": len(final_results),
                "successful_executions": len([r for r in final_results.values() if not str(r).startswith("Error")]),
                "failed_executions": len([r for r in final_results.values() if str(r).startswith("Error")])
            }
        }
        
        if manager_result is None:
            return {
                "query_index": query_index,
                "query_key": query_keys[query_index] if query_index < len(query_keys) else "Unknown",
                "status": "failed",
                "error": "Manager agent returned None"
            }
        
        # Chain of Thought with context - AUTO query assignment
        chain_result = concurrent_call_generate_impact_chain_with_mermaid_auto(
            ticker=ticker,
            verification_links=[],
            verification_reasoning="",
            agent_analysis_results=manager_result['agent_results'],
            total_queries=queries_list,  # Use the list, not the dict
            query_index=query_index,
            context={"specialization_required": True},
            language=language
        )
        
        # Add query key to result
        chain_result["query_key"] = query_keys[query_index] if query_index < len(query_keys) else "Unknown"
        
        return chain_result
        
    except Exception as e:
        return {
            "query_index": query_index,
            "query_key": query_keys[query_index] if query_index < len(query_keys) else "Unknown",
            "status": "failed",
            "error": str(e)
        }

# True multiprocessing - run all queries in parallel
async def run_multiprocessing_pipeline(language: str = "English"):
    # Create tasks for all queries
    tasks = [process_single_query_concurrent(i, language) for i in range(len(queries_list))]
    
    # Run all tasks concurrently
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    return results

# Run true multiprocessing with language setting
final_results = await run_multiprocessing_pipeline(language=language)

# Print results
print("\n🎯 FINAL RESULTS:")
print("=" * 60)
for i, result in enumerate(final_results):
    if isinstance(result, Exception):
        print(f"❌ Query {i} ({query_keys[i] if i < len(query_keys) else 'Unknown'}): Exception - {result}")
    elif result.get("status") == "failed":
        print(f"❌ Query {i} ({result.get('query_key', 'Unknown')}): {result.get('error', 'Unknown error')}")
    else:
        print(f"✅ Query {i} ({result.get('query_key', 'Unknown')}): Success")
        print(f"   Direction: {result.get('chain_of_thought', {}).get('final_direction', 'Unknown')}")

✅ LLM_Call_Agent integration available
✅ Using API keys from LLM_Call_Agent
🤖 Shared Client Pool created (not initialized yet)
✅ LLM_Call_Agent integration available
✅ Using API keys from LLM_Call_Agent
🤖 Shared Client Pool created (not initialized yet)
✅ LLM_Call_Agent integration available
✅ Using API keys from LLM_Call_Agent
🤖 Shared Client Pool created (not initialized yet)
✅ All modules reloaded and imported successfully!
📦 Available functions:
   - quick_analysis
   - get_manager_instance
   - concurrent_call_generate_impact_chain_with_mermaid_auto
📋 Total queries: 6
🔑 Query keys: ['A', 'B', 'C', 'D', 'E', 'F']
📝 Query list: ['分析PLTR当前估值倍数是否过高，是否存在泡沫风险及回调可能性', '评估AI市场情绪波动对PLTR商业前景和股价表现的具体影响机制', '研究经济放缓对政府和企业数据分析支出的实际影响程度', '分析PLTR商业市场扩张进展和收入多元化战略效果', '评估技术突破形态和近期整合后的牛市运行潜力', '分析竞争护城河强度及面对科技巨头和AI初创公司竞争的应对能力']
🌐 Language setting: Chinese
🚀 Initializing Manager Agent for concurrent processing...
🚀 Creating global Manager Agent instance...
🚀 Initializing shared clients for Manager Ag

/Users/xikinki/Desktop/Fintegrate_AI_File/New_Fintegrate_AI(V3)/Stock_Trend_Storage_Agent.py:360: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)


🔄 Process 2: Processing downtrend1
🔄 Process 5: Processing uptrend3
🔄 Process 3: Processing uptrend2
🔄 Process 6: Processing downtrend3
🔄 Process 4: Processing downtrend2
🔄 Process 7: Processing uptrend4
🔄 Process 10: Processing uptrend6
🔄 Process 1: Processing uptrend1
🔄 Process 8: Processing uptrend5
🔄 Process 9: Processing downtrend4
✅ Process 4: Completed downtrend2 with 13 news articles
🔄 Process 11: Processing downtrend5
✅ Process 10: Completed uptrend6 with 13 news articles
🔄 Process 12: Processing uptrend7
✅ Process 6: Completed downtrend3 with 19 news articles
🔄 Process 13: Processing downtrend6
✅ Process 2: Completed downtrend1 with 19 news articles
🔄 Process 14: Processing downtrend7
✅ Process 3: Completed uptrend2 with 19 news articles
🔄 Process 15: Processing uptrend8
✅ Process 7: Completed uptrend4 with 25 news articles
🔄 Process 16: Processing downtrend8
✅ Process 9: Completed downtrend4 with 25 news articles
🔄 Process 17: Processing uptrend9
✅ Process 1: Completed uptre

/Users/xikinki/Desktop/Fintegrate_AI_File/New_Fintegrate_AI(V3)/Stock_Trend_Storage_Agent.py:360: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)


🔄 Process 10: Processing uptrend6
🔄 Process 1: Processing uptrend1
🔄 Process 8: Processing uptrend5
🔄 Process 5: Processing uptrend3
🔄 Process 6: Processing downtrend3
🔄 Process 4: Processing downtrend2
🔄 Process 9: Processing downtrend4
🔄 Process 2: Processing downtrend1
🔄 Process 3: Processing uptrend2
🔄 Process 7: Processing uptrend4
✅ Process 4: Completed downtrend2 with 13 news articles
🔄 Process 11: Processing downtrend5
✅ Process 10: Completed uptrend6 with 13 news articles
🔄 Process 12: Processing uptrend7
✅ Process 3: Completed uptrend2 with 19 news articles
🔄 Process 13: Processing downtrend6
✅ Process 2: Completed downtrend1 with 19 news articles
🔄 Process 14: Processing downtrend7
✅ Process 6: Completed downtrend3 with 19 news articles
🔄 Process 15: Processing uptrend8
✅ Process 5: Completed uptrend3 with 25 news articles
🔄 Process 16: Processing downtrend8
✅ Process 14: Completed downtrend7 with 13 news articles
🔄 Process 17: Processing uptrend9
✅ Process 1: Completed uptr

...

🔄 Process 5: Processing uptrend3
🔄 Process 7: Processing uptrend4
🔄 Process 10: Processing uptrend6
🔄 Process 3: Processing uptrend2
🔄 Process 8: Processing uptrend5
🔄 Process 1: Processing uptrend1
🔄 Process 9: Processing downtrend4
🔄 Process 2: Processing downtrend1
🔄 Process 4: Processing downtrend2
🔄 Process 6: Processing downtrend3
✅ Process 4: Completed downtrend2 with 13 news articles
🔄 Process 11: Processing downtrend5
✅ Process 10: Completed uptrend6 with 13 news articles
🔄 Process 12: Processing uptrend7
✅ Process 3: Completed uptrend2 with 19 news articles
🔄 Process 13: Processing downtrend6
✅ Process 6: Completed downtrend3 with 19 news articles
🔄 Process 14: Processing downtrend7
✅ Process 2: Completed downtrend1 with 19 news articles
🔄 Process 15: Processing uptrend8
✅ Process 9: Completed downtrend4 with 25 news articles
🔄 Process 16: Processing downtrend8
✅ Process 5: Completed uptrend3 with 25 news articles
🔄 Process 17: Processing uptrend9
✅ Process 14: Completed down

/Users/xikinki/Desktop/Fintegrate_AI_File/New_Fintegrate_AI(V3)/Stock_Trend_Storage_Agent.py:360: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)


🔄 Process 5: Processing uptrend3
🔄 Process 10: Processing uptrend6🔄 Process 1: Processing uptrend1

🔄 Process 9: Processing downtrend4
🔄 Process 6: Processing downtrend3
🔄 Process 4: Processing downtrend2
🔄 Process 8: Processing uptrend5
🔄 Process 3: Processing uptrend2
🔄 Process 7: Processing uptrend4
🔄 Process 2: Processing downtrend1
✅ Process 10: Completed uptrend6 with 13 news articles
🔄 Process 11: Processing downtrend5
✅ Process 4: Completed downtrend2 with 13 news articles
🔄 Process 12: Processing uptrend7
✅ Process 3: Completed uptrend2 with 19 news articles
🔄 Process 13: Processing downtrend6
✅ Process 6: Completed downtrend3 with 19 news articles
🔄 Process 14: Processing downtrend7
✅ Process 9: Completed downtrend4 with 25 news articles
🔄 Process 15: Processing uptrend8
✅ Process 14: Completed downtrend7 with 13 news articles
🔄 Process 16: Processing downtrend8
✅ Process 5: Completed uptrend3 with 25 news articles
🔄 Process 17: Processing uptrend9
✅ Process 1: Completed uptr

/Users/xikinki/Desktop/Fintegrate_AI_File/New_Fintegrate_AI(V3)/Stock_Trend_Storage_Agent.py:360: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)


🔄 Process 10: Processing uptrend6
🔄 Process 4: Processing downtrend2
🔄 Process 8: Processing uptrend5
🔄 Process 3: Processing uptrend2
🔄 Process 5: Processing uptrend3
🔄 Process 2: Processing downtrend1
🔄 Process 6: Processing downtrend3
🔄 Process 1: Processing uptrend1
🔄 Process 9: Processing downtrend4
🔄 Process 7: Processing uptrend4
✅ Process 2: Completed downtrend1 with 19 news articles
🔄 Process 11: Processing downtrend5
✅ Process 3: Completed uptrend2 with 19 news articles
🔄 Process 12: Processing uptrend7
✅ Process 10: Completed uptrend6 with 13 news articles
🔄 Process 13: Processing downtrend6
✅ Process 4: Completed downtrend2 with 13 news articles
🔄 Process 14: Processing downtrend7
✅ Process 5: Completed uptrend3 with 25 news articles
🔄 Process 15: Processing uptrend8
✅ Process 9: Completed downtrend4 with 25 news articles
🔄 Process 16: Processing downtrend8
✅ Process 7: Completed uptrend4 with 25 news articles
🔄 Process 17: Processing uptrend9
✅ Process 13: Completed downtr

In [64]:
final_results

[{'query': '分析PLTR当前估值倍数是否过高，是否存在泡沫风险及回调可能性',
  'query_index': 0,
  'status': 'success',
  'chain_of_thought': {'final_direction': 'Short Term Down',
   'impact_chain': 'PLTR当前估值倍数过高分析 → 市场数据显示类似估值过高事件导致-3.078%日平均回报率（参考2025-06-26至2025-07-01） → 极端估值倍数（EV/Sales 320倍，EV/EBITDA 968倍）引发泡沫担忧 → 分析师下调评级和机构减持（参考-1.615%日平均回报率从2024-10-28至2024-11-04） → 短期回调压力',
   'chain_explanation': '基于市场历史数据，类似估值过高事件曾导致PLTR股价出现显著下跌：2025年6月26日至7月1日期间日平均回报率-3.078%，2024年10月28日至11月4日期间日平均回报率-1.615%。当前PLTR的极端估值倍数（EV/Sales 320倍，EV/EBITDA 968倍）远超行业正常水平，结合分析师下调评级和机构减持的历史模式，预计将引发短期回调压力。',
   'node_count': 5,
   'edge_count': 4,
   'events': ['PLTR当前估值倍数过高分析',
    '市场数据显示类似估值过高事件导致-3.078%日平均回报率（参考2025-06-26至2025-07-01）',
    '极端估值倍数（EV/Sales 320倍，EV/EBITDA 968倍）引发泡沫担忧',
    '分析师下调评级和机构减持（参考-1.615%日平均回报率从2024-10-28至2024-11-04）',
    '短期回调压力']},
  'mermaid_code': 'graph LR\n    title["Chain 5: PLTR当前估值倍数过高分析..."]\n    A[PLTR当前估值倍数过高分析]\n    B[市场数据显示类似估值过高事件导致-3.078日平均回报率参考2025-06-26至2025-0...]\n    C[极端估值倍数EVSales 320倍EVEB

In [65]:
# More detailed extraction with error handling
COT_prepare_Dynamic_Rating = []

for i, result in enumerate(final_results):
    if result.get('status') == 'success':
        chain_data = result.get('chain_of_thought', {})
        if chain_data:
            # Ensure all required fields exist
            chain_result = {
                'ticker': chain_data.get('ticker', ticker),
                'impact_chain': chain_data.get('impact_chain', ''),
                'final_direction': chain_data.get('final_direction', ''),
                'chain_explanation': chain_data.get('chain_explanation', ''),

            }
            COT_prepare_Dynamic_Rating.append(chain_result)
            print(f"✅ Added chain {i+1}: {chain_result['final_direction']}")
        else:
            print(f"⚠️ No chain_of_thought data for result {i+1}")

print(f"📊 Total chain results extracted: {len(COT_prepare_Dynamic_Rating)}")

COT_prepare_Dynamic_Rating

✅ Added chain 1: Short Term Down
✅ Added chain 2: Short Term Down
✅ Added chain 3: Short Term Down
✅ Added chain 4: Long Term Up
✅ Added chain 5: Long Term Up
✅ Added chain 6: Long Term Up
📊 Total chain results extracted: 6


[{'ticker': 'PLTR',
  'impact_chain': 'PLTR当前估值倍数过高分析 → 市场数据显示类似估值过高事件导致-3.078%日平均回报率（参考2025-06-26至2025-07-01） → 极端估值倍数（EV/Sales 320倍，EV/EBITDA 968倍）引发泡沫担忧 → 分析师下调评级和机构减持（参考-1.615%日平均回报率从2024-10-28至2024-11-04） → 短期回调压力',
  'final_direction': 'Short Term Down',
  'chain_explanation': '基于市场历史数据，类似估值过高事件曾导致PLTR股价出现显著下跌：2025年6月26日至7月1日期间日平均回报率-3.078%，2024年10月28日至11月4日期间日平均回报率-1.615%。当前PLTR的极端估值倍数（EV/Sales 320倍，EV/EBITDA 968倍）远超行业正常水平，结合分析师下调评级和机构减持的历史模式，预计将引发短期回调压力。'},
 {'ticker': 'PLTR',
  'impact_chain': 'AI市场情绪波动 → 历史数据显示类似AI情绪事件导致-9%回报率(参考2024年AI板块抛售期间) → 投资者风险偏好下降(-15%估值倍数压缩) → 机构资金流出(-1.615%日平均回报率从2024-10-28至2024-11-04) → 短期商业决策延迟(影响93%增长的美国商业业务) → 短期股价压力',
  'final_direction': 'Short Term Down',
  'chain_explanation': '基于历史数据，AI市场情绪波动导致PLTR股价在类似事件中下跌-9%，参考2024年AI板块抛售期间的表现。从2024-10-28至2024-11-04期间，PLTR出现-1.615%的日平均回报率，显示机构资金流出压力。虽然公司基本面强劲(48%收入增长，93%美国商业增长)，但短期情绪波动影响投资者风险偏好和估值倍数。'},
 {'ticker': 'PLTR',
  'impact_chain': '经济放缓 → 市场数据显示类似经济不确定性事件导致-2.927%回报率（参考2025-08-12至2025-08-20） → 

# Section 3). Conclusion AI 

In [66]:
import asyncio
import json
from typing import Dict, Any
from pydantic import BaseModel, Field
from langchain_deepseek import ChatDeepSeek
import os

# Set API key
os.environ["DEEPSEEK_API_KEY"] = "sk-43e9043c7ab8480393d34367f2ae997e"

# Define the Pydantic model
class ChainOfThoughtConclusionResult(BaseModel):
    short_term_impact: str = Field(description="Detailed analysis of short-term impact (50 words with clear safe or not safe in current position price)")
    long_term_outlook: str = Field(description="Detailed analysis of long-term outlook (50 words with clear safe or not safe in current position price)")
    Dynamic_Rating: str = Field(description="if over come the short term risk, what is the long term reward? (50 words) or if not over come the short term risk, what is the long term risk? (50 words)")
    Catalyst: str = Field(description="From all u have read and analyst what id the three things that chanage (in the chains), then the whole story change (50 words)")


class ChainOfThoughtConclusionAgent:
    def __init__(self):
        """Initialize the Chain of Thought Conclusion Agent with direct LLM initialization"""
        self.llm_agent = None
        self.structured_llm = None
        
        # Direct LLM initialization (same as Supervisor Agent)
        try:
            from LLM_Call_Agent import DEEPSEEK_API_KEY
            self.llm_agent = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
                api_key=DEEPSEEK_API_KEY
            )
        except ImportError:
            self.llm_agent = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
            )
        
        if self.llm_agent is None:
            raise Exception("❌ CRITICAL ERROR: LLM Call Agent is None. Check API keys and network connection.")
        
        # Create structured LLM
        self.structured_llm = self.llm_agent.with_structured_output(ChainOfThoughtConclusionResult)
        print("✅ Chain of Thought Conclusion Agent initialized with direct LLM")
    
    async def analyze_impact(self, business_logic: str, chain_of_thought: str, language: str = "English") -> Dict[str, Any]:
        """
        Analyze short-term vs long-term impact based on business logic and chain of thought
        
        Args:
            business_logic: The business logic/moat analysis
            chain_of_thought: The current sell/buy chain of thought result
            language: Language for output
            
        Returns:
            Dictionary with analysis results
        """
        
        prompt = f"""
        You are an expert financial analyst specializing in short-term vs long-term impact analysis.

        TASK: Analyze the impact on the company based on business logic (moat) and current sell/buy chain of thought.

        BUSINESS LOGIC (MOAT): {business_logic}

        CHAIN OF THOUGHT RESULT: {chain_of_thought}

        ANALYSIS FRAMEWORK:
        1. **SHORT-TERM IMPACT ANALYSIS**:
           - Identify immediate challenges, troubles, or potential bubbles
           - Assess market sentiment and short-term headwinds
           - Evaluate temporary vs structural issues
           - Consider cyclical vs secular factors

        2. **LONG-TERM OUTLOOK ANALYSIS**:
           - Evaluate company's ability to overcome short-term troubles or maintain the business sustainability (success)
           - Assess competitive advantages and moat sustainability
           - Analyze strategic positioning for long-term success
           - Consider fundamental business model strength

        3. **DYNAMIC RATING ANALYSIS**:
           - Assess short-term risk vs reward
           - Assess long-term risk vs reward
           - Compare short-term vs long-term perspectives

        4. **CATALYST IDENTIFICATION**:
           - Identify the three key things that could change the whole story
           - Focus on factors that would significantly alter the chain of thought

        LANGUAGE REQUIREMENT: All output must be in {language}. Respond entirely in {language}.

        OUTPUT REQUIREMENTS:
        - Be specific and data-driven
        - Distinguish between temporary and permanent factors
        - Provide clear investment implications
        - Use professional financial analysis language
        - Keep responses concise but comprehensive
        - Provide reward scores as decimal numbers (0.0 to 1.0)
        """
        
        try:
            # Use structured LLM invoke method
            result = self.structured_llm.invoke(prompt)
            return result.model_dump()  # Convert Pydantic model to dict (v2 syntax)
            
        except Exception as e:
            raise Exception(f"Chain of Thought Conclusion analysis failed: {e}")

# Convenience function with global language detection
async def analyze_chain_conclusion(business_logic: str, chain_of_thought: str, language: str = None) -> Dict[str, Any]:
    """
    Convenience function to analyze chain of thought conclusion
    
    Args:
        business_logic: The business logic/moat analysis
        chain_of_thought: The current sell/buy chain of thought result
        language: Language for output (if None, will use global language variable)
        
    Returns:
        Dictionary with analysis results
    """
    try:
        # Use global language variable if not provided
        if language is None:
            try:
                language = globals()['language']
                print(f"🌐 Using global language setting: {language}")
            except KeyError:
                language = "English"
                print("⚠️ Global language variable not found, using default: English")
        
        agent = ChainOfThoughtConclusionAgent()
        result = await agent.analyze_impact(business_logic, chain_of_thought, language)
        return result
    except Exception as e:
        print(f"❌ Error in analyze_chain_conclusion: {e}")
        return {
            "short_term_impact": f"Analysis failed: {e}",
            "long_term_outlook": "Analysis failed",
            "Dynamic_Rating": "Analysis failed", 
            "Catalyst": "Analysis failed",
            "Short_term_reward": 0.5,
            "Long_term_reward": 0.5
        }

In [67]:
Conclusion_Result = await analyze_chain_conclusion(business_logic, final_results)

🌐 Using global language setting: Chinese
✅ Chain of Thought Conclusion Agent initialized with direct LLM
2025-09-09 11:13:47,431 - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


In [68]:
Conclusion_Result

{'short_term_impact': '基于三个短期下行链条分析，PLTR在当前价位面临显著风险。极端估值倍数（EV/Sales 320倍，EV/EBITDA 968倍）超过行业正常水平，历史数据显示类似估值过高事件导致-3.078%日平均回报率。AI市场情绪波动和经济放缓增加了短期回调压力，当前价位不安全。',
 'long_term_outlook': '长期看好PLTR的业务可持续性。双核心商业模式与20年CIA背景极具竞争优势，客户粘性强且切换成本高。商业收入增长71% YoY，政府收入占比降至55%，多元化战略成功。Rule of 40得分94%显示质量高增长，当前价位长期安全。',
 'Dynamic_Rating': '如果成功渗透短期估值风险，长期奖励极具吸引力。估值回归合理水平后，基于48%收入增长和93%美国商业收入增长，长期回报潜力达523.5%。企业AI平台领导地位与双重市场战略极具长期价值。',
 'Catalyst': '三个关键变化点：1）估值回归合理水平后的投资者信心恢复；2）AI市场情绪稳定与经济环境改善；3）商业收入持续高增长确认多元化战略成功。这些变化将改变整体投资敏感性。'}

# 4). Dynamic Rating AI 

In [69]:
import os
from typing import Dict, Any, List
from pydantic import BaseModel, Field
from langchain_deepseek import ChatDeepSeek

# Set API key
os.environ["DEEPSEEK_API_KEY"] = "sk-43e9043c7ab8480393d34367f2ae997e"

# Define the Pydantic model for Dynamic Rating with reasoning
class DynamicRatingResult(BaseModel):
    ShortTermRisk: float = Field(description="Short-term risk score (0.0 to 1.0)")
    ShortTermReward: float = Field(description="Short-term reward score (0.0 to 1.0)")
    LongTermRisk: float = Field(description="Long-term risk score (0.0 to 1.0)")
    LongTermReward: float = Field(description="Long-term reward score (0.0 to 1.0)")
    Reasoning: str = Field(description="Detailed explanation of how the scores were calculated, including price pattern analysis")

class DynamicRatingAgent:
    def __init__(self):
        """Initialize the Dynamic Rating Agent with direct LLM initialization"""
        self.llm_agent = None
        self.structured_llm = None
        
        # Direct LLM initialization
        try:
            from LLM_Call_Agent import DEEPSEEK_API_KEY
            self.llm_agent = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
                api_key=DEEPSEEK_API_KEY
            )
        except ImportError:
            self.llm_agent = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
            )
        
        if self.llm_agent is None:
            raise Exception("❌ CRITICAL ERROR: LLM Call Agent is None. Check API keys and network connection.")
        
        # Create structured LLM
        self.structured_llm = self.llm_agent.with_structured_output(DynamicRatingResult)
        print("✅ Dynamic Rating Agent initialized with direct LLM")
    
    async def call_stock_trend_agent(self, ticker: str) -> str:
        """
        Call Stock Trend Agent to get current price level and recent events
        
        Args:
            ticker: Stock ticker symbol
            
        Returns:
            String with stock trend analysis
        """
        try:
            # Import the correct Stock Trend Agent
            from Stock_Trend_Read_Agent import StockTrendAnalystAgent
            
            # Initialize the agent with proper Redis connection details
            stock_trend_agent = StockTrendAnalystAgent(
                redis_host="redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com",
                redis_port=16376,
                redis_username="default",
                redis_password="rl8242B4UItBhFzgHW5APEqZnkYoaEZv"
            )
            
            # Create query for current price level and recent events
            query = f"""
            Analyze {ticker} current price level and recent events:
            has the current sentiment and events already priced in?
            what level they price in?
           
            """
            
            # Call the agent with the correct method name
            result = await stock_trend_agent.process_natural_query(query, ticker)
            
            # Extract the analysis text
            if isinstance(result, dict):
                analysis_text = result.get('analysis', str(result))
            else:
                analysis_text = str(result)
            
            print(f"✅ Stock Trend Agent called for {ticker}")
            return analysis_text
            
        except Exception as e:
            print(f"⚠️ Error calling Stock Trend Agent: {e}")
            return f"Stock trend analysis unavailable for {ticker}: {e}"
    
    async def generate_dynamic_rating(
        self, 
        chain_of_thought_results: List[Dict[str, Any]], 
        conclusion_result: Dict[str, Any],
        ticker: str,
        language: str = "English"
    ) -> Dict[str, Any]:
        """
        Generate dynamic rating scores based on chain of thought, conclusion results, and stock trend analysis
        
        Args:
            chain_of_thought_results: List of chain of thought results
            conclusion_result: Conclusion analysis result
            ticker: Stock ticker symbol
            language: Language for output
            
        Returns:
            Dictionary with dynamic rating scores and reasoning
        """
        
        # Call Stock Trend Agent first
        print(f"📊 Calling Stock Trend Agent for {ticker}...")
        stock_trend_analysis = await self.call_stock_trend_agent(ticker)
        
        # Extract key information from conclusion result
        short_term_impact = conclusion_result.get('short_term_impact', '')
        long_term_outlook = conclusion_result.get('long_term_outlook', '')
        dynamic_rating = conclusion_result.get('Dynamic_Rating', '')
        catalyst = conclusion_result.get('Catalyst', '')
        
        # Extract chain information
        chain_summary = ""
        for i, chain in enumerate(chain_of_thought_results):
            direction = chain.get('final_direction', '')
            chain_text = chain.get('impact_chain', '')
            chain_summary += f"Chain {i+1}: {direction} - {chain_text}\n"
        
        prompt = f"""
        You are a financial analyst AI. Assign 4 scores (0.0 to 1.0 scale):
        - ShortTermRisk, ShortTermReward, LongTermRisk, LongTermReward

        SCORING CRITERIA:

        **SHORT TERM RISK/REWARD (Current Price Level Consideration):**
        - If event is already priced in → Risk LOWER (0.2-0.4), Reward LOWER (0.2-0.4)
        - If event not yet priced in → Risk HIGHER (0.6-0.8), Reward HIGHER (0.6-0.8)
        - Signal strength: ≤3% = 0.3-0.4, 3-10% = 0.4-0.6, 10-20% = 0.6-0.8, >20% = 0.8-1.0

        **LONG TERM RISK/REWARD (Historical Data + Future Trends):**
        - Based on historical patterns and future implications from chains
        - Future trend pressure/difficulty to maintain success/overcome challenges
        - If chains show future difficulties → LongTermRisk HIGHER (0.6-0.8)
        - If chains show future opportunities → LongTermReward HIGHER (0.6-0.8)

        **HORIZON RULES:**
        - Short Term (<180 days) → ShortTermRisk/Reward
        - Long Term (≥180 days) → LongTermRisk/Reward

        **ADJUSTMENT FACTORS:**
        - Catalyst: Positive → reduce risk 0.1-0.2, increase reward 0.1-0.2
        - Catalyst: Negative → increase risk 0.1-0.2, reduce reward 0.1-0.2

        These risk  and reward should not sum up to 1, just make they very reason, dont 
        always come out with a pair of something, please stricly follow the rules
        again they should not necesarily sum up to 1, just make them very reasonable

        INPUT DATA:
        - Ticker: {ticker}
        - Short Term Impact: {short_term_impact}
        - Long Term Outlook: {long_term_outlook}
        - Dynamic Rating: {dynamic_rating}
        - Catalyst: {catalyst}
        - Stock Trend Analysis: {stock_trend_analysis}
        - Chain of Thought Results: {chain_summary}

        OUTPUT FORMAT:
        {{
          "ShortTermRisk": <float 0.0-1.0>,
          "ShortTermReward": <float 0.0-1.0>,
          "LongTermRisk": <float 0.0-1.0>,
          "LongTermReward": <float 0.0-1.0>,
          "Reasoning": "Brief explanation: 1) Current price level impact on short-term scores, 2) Historical data and future trend implications for long-term scores, 3) Signal strength assessment"
        }}

        IMPORTANT:
        - Consider if events are already priced in for short-term scores
        - Use historical data and future trend implications for long-term scores
        - Factor in pressure/difficulty to maintain success from chains

        please output in language: {language} for me, as you are api call that might handle different language in default
        """
        
        try:
            result = self.structured_llm.invoke(prompt)
            return result.model_dump()  # Convert Pydantic model to dict (v2 syntax)
        except Exception as e:
            raise Exception(f"Dynamic Rating analysis failed: {e}")

# Convenience function with global language detection
async def generate_dynamic_rating_scores(
    chain_of_thought_results: List[Dict[str, Any]], 
    conclusion_result: Dict[str, Any],
    ticker: str,
    language: str = None
) -> Dict[str, Any]:
    """
    Convenience function to generate dynamic rating scores with stock trend analysis
    
    Args:
        chain_of_thought_results: List of chain of thought results
        conclusion_result: Conclusion analysis result
        ticker: Stock ticker symbol
        language: Language for output (if None, will use global language variable)
        
    Returns:
        Dictionary with dynamic rating scores and reasoning
    """
    try:
        # Use global language variable if not provided
        if language is None:
            try:
                language = globals()['language']
                print(f"🌐 Using global language setting: {language}")
            except KeyError:
                language = "English"
                print("⚠️ Global language variable not found, using default: English")
        
        agent = DynamicRatingAgent()
        result = await agent.generate_dynamic_rating(chain_of_thought_results, conclusion_result, ticker, language)
        return result
    except Exception as e:
        print(f"❌ Error in generate_dynamic_rating_scores: {e}")
        return {
            "ShortTermRisk": 0.5,
            "ShortTermReward": 0.5,
            "LongTermRisk": 0.5,
            "LongTermReward": 0.5,
            "Reasoning": f"Error occurred: {e}"
        }

In [70]:
dynamic_Rating =  await generate_dynamic_rating_scores(COT_prepare_Dynamic_Rating, Conclusion_Result, ticker, language)



✅ Dynamic Rating Agent initialized with direct LLM
📊 Calling Stock Trend Agent for PLTR...
2025-09-09 11:14:45,181 - INFO - Attempting to connect to Redis...
2025-09-09 11:14:45,182 - INFO - Host: redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com
2025-09-09 11:14:45,182 - INFO - Port: 16376
2025-09-09 11:14:45,182 - INFO - Username: default
2025-09-09 11:14:45,183 - INFO - Testing connection with ping command...
2025-09-09 11:14:45,473 - INFO - ✓ Ping successful - Redis server is reachable
2025-09-09 11:14:45,474 - INFO - ✓ Successfully connected to Redis
2025-09-09 11:14:45,474 - INFO - ✅ Using shared LLM client
2025-09-09 11:14:45,475 - INFO - 🤖 Stock Trend Analyst Agent initialized
2025-09-09 11:14:45,475 - INFO -    - Redis: redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com:16376
2025-09-09 11:14:45,475 - INFO -    - Collection: Stock_Trend_INFOS
2025-09-09 11:14:45,475 - INFO -    - LLM Provider: {'available': True, 'client_initialized': True, 'api_key': '✅ Availab

In [71]:
dynamic_Rating

{'ShortTermRisk': 0.75,
 'ShortTermReward': 0.25,
 'LongTermRisk': 0.35,
 'LongTermReward': 0.85,
 'Reasoning': '1) 短期风险高：当前极端估值倍数（EV/Sales 320倍，EV/EBITDA 968倍）远超行业正常水平，历史数据显示类似估值过高事件导致-3.078%日平均回报率，AI市场情绪波动和经济放缓增加回调压力，事件尚未完全定价，信号强度约15-20%；2) 长期前景乐观：基于历史扩张事件+19.46%回报和技术突破+23.5%回报的强劲表现，PLTR商业收入增长71% YoY，政府收入占比降至55%，Rule of 40得分94%显示高质量增长，竞争护城河强大；3) 催化剂因素：估值回归合理、AI情绪稳定和商业收入持续增长将显著改善投资敏感性，长期回报潜力达523.5%。'}

### Paramter Summarize from the whole pipeline 

In [72]:
print(internet_search_result)
print(final_results )
print(Conclusion_Result)
print(COT_prepare_Dynamic_Rating)
print(dynamic_Rating)




total_sub_managers=6 sub_managers=[SubManagerQuery(manager_id='A', query='分析PLTR当前估值倍数是否过高，是否存在泡沫风险及回调可能性', dimension='估值风险', priority=9, expected_impact='Negative'), SubManagerQuery(manager_id='B', query='评估AI市场情绪波动对PLTR商业前景和股价表现的具体影响机制', dimension='市场情绪', priority=8, expected_impact='Mixed'), SubManagerQuery(manager_id='C', query='研究经济放缓对政府和企业数据分析支出的实际影响程度', dimension='经济敏感性', priority=7, expected_impact='Negative'), SubManagerQuery(manager_id='D', query='分析PLTR商业市场扩张进展和收入多元化战略效果', dimension='增长潜力', priority=8, expected_impact='Positive'), SubManagerQuery(manager_id='E', query='评估技术突破形态和近期整合后的牛市运行潜力', dimension='技术分析', priority=6, expected_impact='Positive'), SubManagerQuery(manager_id='F', query='分析竞争护城河强度及面对科技巨头和AI初创公司竞争的应对能力', dimension='竞争优势', priority=7, expected_impact='Mixed')] buy_side_summary=['AI平台驱动80%合同，提供20-30%效率增益', '美国政府收入增长45%，商业收入增长71%', '客户数量增长39%至498家，大额交易显著增加', '净美元留存率128%，运营利润率46%', '技术突破形态显示牛市运行潜力', '商业市场扩张推动收入多元化'] sell_side_summary=['估值倍数极高，存在泡沫风险', '经济放缓可能影响政

## Section 4). Final Result in Frontend

In [73]:
# Q&Q.AI Report-Style Dashboard - Large Logo Design
import webbrowser
import os
from datetime import datetime

def visualize_qq_ai_report():
    """
    Report-style visualization: Chains first, then summary paragraphs
    Single horizontal bar with two segments for rewards
    Added color legend with language support
    Two-column layout for sell/buy side analysis
    Updated with Dynamic Rating structure
    Higher scores = Green, Lower scores = Red
    Large logo design with invisible background
    """
    
    # Process final_results to get chain data
    chain_data = []
    
    for i, result_item in enumerate(final_results):
        if result_item.get('status') == 'success':
            chain_info = result_item.get('chain_of_thought', {})
            events = chain_info.get('events', [])
            
            # Extract starting event for title
            starting_event = events[0] if events else "Unknown starting event"
            
            chain_data.append({
                'index': i + 1,
                'query_key': result_item.get('query_key', f'Query {i+1}'),
                'direction': chain_info.get('final_direction', 'Unknown'),
                'chain': chain_info.get('impact_chain', 'Unknown'),
                'mermaid_code': result_item.get('mermaid_code', ''),
                'starting_event': starting_event
            })
    
    # Get data from internet_search_result - SEPARATE SELL/BUY SIDES
    try:
        sell_side_data = internet_search_result.sell_side_summary
        buy_side_data = internet_search_result.buy_side_summary
        business_logic_data = internet_search_result.business_logic
        print(f"✅ Successfully extracted from internet_search_result:")
        print(f"   - Sell side items: {len(sell_side_data)}")
        print(f"   - Buy side items: {len(buy_side_data)}")
        print(f"   - Business logic items: {len(business_logic_data)}")
    except NameError as e:
        print(f"⚠️ internet_search_result not found: {e}")
        sell_side_data = []
        buy_side_data = []
        business_logic_data = "No business logic available"
    
    # Get conclusion results
    dynamic_rating = Conclusion_Result.get('Dynamic_Rating', 'Not available') if 'Conclusion_Result' in globals() else 'Not available'
    catalyst = Conclusion_Result.get('Catalyst', 'Not available') if 'Conclusion_Result' in globals() else 'Not available'
    short_term_impact = Conclusion_Result.get('short_term_impact', 'Not available') if 'Conclusion_Result' in globals() else 'Not available'
    long_term_outlook = Conclusion_Result.get('long_term_outlook', 'Not available') if 'Conclusion_Result' in globals() else 'Not available'
    
    # Get Dynamic Rating scores - NEW STRUCTURE
    try:
        # Parse dynamic_rating variable (assuming it's a dict with the four scores)
        if 'dynamic_Rating' in globals():
            dynamic_rating_data = dynamic_Rating
            short_term_risk = dynamic_rating_data.get('ShortTermRisk', 0.5)
            short_term_reward = dynamic_rating_data.get('ShortTermReward', 0.5)
            long_term_risk = dynamic_rating_data.get('LongTermRisk', 0.5)
            long_term_reward = dynamic_rating_data.get('LongTermReward', 0.5)
            dynamic_reasoning = dynamic_rating_data.get('Reasoning', 'No reasoning available')
            
            # Ensure they are floats
            short_term_risk = float(short_term_risk)
            short_term_reward = float(short_term_reward)
            long_term_risk = float(long_term_risk)
            long_term_reward = float(long_term_reward)
            
            print(f"✅ Dynamic Rating scores extracted:")
            print(f"   - Short Term Risk: {short_term_risk}")
            print(f"   - Short Term Reward: {short_term_reward}")
            print(f"   - Long Term Risk: {long_term_risk}")
            print(f"   - Long Term Reward: {long_term_reward}")
            
        else:
            raise ValueError("dynamic_Rating variable not found")
            
    except (KeyError, ValueError, TypeError) as e:
        print(f"❌ Failed to extract Dynamic Rating scores: {e}")
        # Use default values
        short_term_risk = 0.5
        short_term_reward = 0.5
        long_term_risk = 0.5
        long_term_reward = 0.5
        dynamic_reasoning = "Dynamic Rating analysis not available"
    
    # Convert lists to HTML paragraphs
    def list_to_paragraphs(items):
        if isinstance(items, list):
            return "".join([f"<p>{item}</p>\n" for item in items])
        else:
            return f"<p>{items}</p>\n"
    
    sell_side_html = list_to_paragraphs(sell_side_data)
    buy_side_html = list_to_paragraphs(buy_side_data)
    
    # Generate chain cards HTML
    chain_cards_html = ""
    for chain in chain_data:
        # Determine direction class
        direction_class = "unknown"
        if "Short Term Up" in chain['direction']:
            direction_class = "short-term-up"
        elif "Short Term Down" in chain['direction']:
            direction_class = "short-term-down"
        elif "Long Term Up" in chain['direction']:
            direction_class = "long-term-up"
        elif "Long Term Down" in chain['direction']:
            direction_class = "long-term-down"
        
        chain_cards_html += f"""
            <div class="chain-card">
                <div class="chain-header">
                    <div class="chain-title-wrapper">
                        <div class="chain-number">{chain['index']}</div>
                        <div class="chain-title">{chain['starting_event']}</div>
                    </div>
                    <span class="direction-badge {direction_class}">{chain['direction']}</span>
                </div>
                <div class="chain-content">{chain['chain']}</div>
                <div class="mermaid-wrapper">
                    <div class="mermaid">{chain['mermaid_code']}</div>
                </div>
            </div>
        """
    
    # NEW: Color scheme - Higher scores = Green, Lower scores = Red
    def get_score_class(score, is_risk=False):
        """
        For Risk: Lower is better (green), Higher is worse (red)
        For Reward: Higher is better (green), Lower is worse (red)
        """
        if is_risk:
            # For risk: lower scores are better (green)
            if score <= 0.3:
                return "excellent"  # Very low risk = green
            elif score <= 0.5:
                return "good"       # Low risk = light green
            elif score <= 0.7:
                return "warning"    # Medium risk = yellow
            else:
                return "danger"     # High risk = red
        else:
            # For reward: higher scores are better (green)
            if score >= 0.7:
                return "excellent"  # High reward = green
            elif score >= 0.5:
                return "good"       # Medium reward = light green
            elif score >= 0.3:
                return "warning"    # Low reward = yellow
            else:
                return "danger"     # Very low reward = red
    
    short_term_risk_class = get_score_class(short_term_risk, is_risk=True)
    long_term_risk_class = get_score_class(long_term_risk, is_risk=True)
    short_term_reward_class = get_score_class(short_term_reward, is_risk=False)
    long_term_reward_class = get_score_class(long_term_reward, is_risk=False)
    
    # Get ticker and other missing variables
    try:
        ticker_symbol = ticker
    except NameError:
        ticker_symbol = "UNKNOWN"
    
    # Generate dates
    analysis_date = datetime.now().strftime("%Y-%m-%d")
    report_date_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    chain_count = len(chain_data)
    
    # Language detection and text generation
    try:
        global_language = globals()['language']
        is_chinese = global_language.lower() == 'chinese'
    except KeyError:
        is_chinese = False
    
    # Generate language-specific text - UPDATED TITLES
    if is_chinese:
        color_legend_title = "影响链方向说明"
        color_legend_subtitle = "不同颜色代表不同的投资方向和时间维度"
        long_term_up_text = "长期上涨"
        long_term_down_text = "长期下跌"
        short_term_up_text = "短期上涨"
        short_term_down_text = "短期下跌"
        impact_chains_title = "定性AI"  # NEW TITLE
        sell_buy_title = "买卖分析总结"
        sell_side_title = "卖出分析"
        buy_side_title = "买入分析"
        business_logic_title = "商业逻辑总结"
        conclusion_title = "定量AI"  # NEW TITLE
        short_term_risk_text = "短期风险评分"
        long_term_risk_text = "长期风险评分"
        short_term_reward_text = "短期回报评分"
        long_term_reward_text = "长期回报评分"
        dynamic_rating_text = "动态评级"
        key_catalysts_text = "关键催化剂"
        short_term_impact_text = "短期影响"
        long_term_outlook_text = "长期展望"
        dynamic_reasoning_title = "AI风险评级分析说明"
        footer_text = "由 Q&Q.AI  - 定量&定性 AI"
        report_generated_text = "报告生成时间"
        copyright_text = "© 2025 Q&Q.AI - 连接数据智能"
        logo_description = "定量与定性AI投资分析系统"
    else:
        color_legend_title = "Impact Chain Direction Legend"
        color_legend_subtitle = "Different colors represent different investment directions and time horizons"
        long_term_up_text = "Long Term Up"
        long_term_down_text = "Long Term Down"
        short_term_up_text = "Short Term Up"
        short_term_down_text = "Short Term Down"
        impact_chains_title = "Qualitative AI"  # NEW TITLE
        sell_buy_title = "Sell & Buy Analysis Summary"
        sell_side_title = "Sell Side Analysis"
        buy_side_title = "Buy Side Analysis"
        business_logic_title = "Business Logic Summary"
        conclusion_title = "Quantitative AI"  # NEW TITLE
        short_term_risk_text = "Short Term Risk Score"
        long_term_risk_text = "Long Term Risk Score"
        short_term_reward_text = "Short Term Reward Score"
        long_term_reward_text = "Long Term Reward Score"
        dynamic_rating_text = "Dynamic Rating"
        key_catalysts_text = "Key Catalysts"
        short_term_impact_text = "Short Term Impact"
        long_term_outlook_text = "Long Term Outlook"
        dynamic_reasoning_title = "Dynamic Rating Analysis Explanation"
        footer_text = "Generated by Q&Q.AI - Quantitative & Qualitative AI Investment Analysis System"
        report_generated_text = "Report generated on"
        copyright_text = "© 2025 Q&Q.AI - Bridging Data Intelligence"
        logo_description = "Quantitative & Qualitative AI Investment Analysis System"
    
    # Generate color legend HTML
    color_legend_html = f"""
        <div class="color-legend">
            <h3 class="legend-title">{color_legend_title}</h3>
            <p class="legend-subtitle">{color_legend_subtitle}</p>
            <div class="legend-grid">
                <div class="legend-item">
                    <div class="legend-color long-term-up"></div>
                    <span class="legend-text">{long_term_up_text}</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color long-term-down"></div>
                    <span class="legend-text">{long_term_down_text}</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color short-term-up"></div>
                    <span class="legend-text">{short_term_up_text}</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color short-term-down"></div>
                    <span class="legend-text">{short_term_down_text}</span>
                </div>
            </div>
        </div>
    """
    
    # Generate section titles with rings
    def get_section_title_with_ring(title_text, is_quantitative=True):
        ring_class = "ring-quantitative" if is_quantitative else "ring-qualitative"
        ring_glow_class = "ring-glow-quantitative" if is_quantitative else "ring-glow-qualitative"
        
        return f"""
            <div class="section-title-with-ring">
                <div class="single-ring {ring_class}">
                    <div class="ring-glow {ring_glow_class}"></div>
                </div>
                <div class="section-title-text">{title_text}</div>
            </div>
        """
    
    # Generate section titles
    impact_chains_title_html = get_section_title_with_ring(impact_chains_title, is_quantitative=False)
    conclusion_title_html = get_section_title_with_ring(conclusion_title, is_quantitative=True)
    
    # Generate report-style HTML - LARGE LOGO DESIGN
    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Q&Q.AI - Impact Chain Analysis Results</title>
    <script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}
        
        body {{
            font-family: 'Courier New', 'Monaco', 'Menlo', monospace;
            background: linear-gradient(135deg, #0f0f23 0%, #1a1a2e 50%, #16213e 100%);
            color: #ffffff;
            min-height: 100vh;
            position: relative;
            overflow-x: hidden;
        }}

        .main-container {{
            position: relative;
            z-index: 2;
            max-width: 1400px;
            margin: 0 auto;
            padding: 40px 20px;
        }}

        /* Large Logo Section - Invisible Background */
        .logo-section {{
            text-align: center;
            margin-bottom: 60px;
            padding: 60px 20px;
            /* NO BACKGROUND - Invisible chunk */
            background: transparent;
            border: none;
            border-radius: 0;
        }}

        .logo-svg {{
            width: 500px;
            height: 200px;
            filter: drop-shadow(0 0 40px rgba(102, 126, 234, 0.8));
            animation: pulse-glow 4s ease-in-out infinite;
        }}

        @keyframes pulse-glow {{
            0%, 100% {{ filter: drop-shadow(0 0 40px rgba(102, 126, 234, 0.8)); }}
            50% {{ filter: drop-shadow(0 0 60px rgba(118, 75, 162, 1)); }}
        }}

        .logo-text {{
            font-size: 48px;
            font-weight: 200;
            color: #ffffff;
            letter-spacing: 12px;
            margin-top: 30px;
            text-shadow: 0 0 30px rgba(102, 126, 234, 0.8);
        }}

        .logo-description {{
            font-size: 20px;
            font-weight: 100;
            color: #a0aec0;
            letter-spacing: 4px;
            margin-top: 15px;
            opacity: 0.9;
        }}

        .ticker-bar {{
            background: rgba(255, 255, 255, 0.02);
            backdrop-filter: blur(20px);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 20px;
            padding: 20px 40px;
            margin-bottom: 40px;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}

        .ticker-symbol {{
            font-size: 2em;
            font-weight: 600;
            color: #667eea;
            text-shadow: 0 0 20px rgba(102, 126, 234, 0.5);
        }}

        .section {{
            margin-bottom: 60px;
        }}

        /* Single Ring Design */
        .ring-container {{
            display: inline-flex;
            align-items: center;
            gap: 15px;
        }}

        .single-ring {{
            width: 25px;
            height: 25px;
            border: 2px solid;
            border-radius: 50%;
            position: relative;
            animation: ring-pulse 2s ease-in-out infinite;
        }}

        .ring-quantitative {{
            border-color: #667eea;
            box-shadow: 0 0 15px rgba(102, 126, 234, 0.6);
        }}

        .ring-qualitative {{
            border-color: #764ba2;
            box-shadow: 0 0 15px rgba(118, 75, 162, 0.6);
        }}

        @keyframes ring-pulse {{
            0%, 100% {{ 
                transform: scale(1);
                opacity: 0.8;
            }}
            50% {{ 
                transform: scale(1.1);
                opacity: 1;
            }}
        }}

        .ring-glow {{
            position: absolute;
            top: -2px;
            left: -2px;
            right: -2px;
            bottom: -2px;
            border-radius: 50%;
            opacity: 0.3;
            animation: ring-glow 3s ease-in-out infinite;
        }}

        .ring-glow-quantitative {{
            border: 1px solid #667eea;
            box-shadow: 0 0 20px rgba(102, 126, 234, 0.4);
        }}

        .ring-glow-qualitative {{
            border: 1px solid #764ba2;
            box-shadow: 0 0 20px rgba(118, 75, 162, 0.4);
        }}

        @keyframes ring-glow {{
            0%, 100% {{ opacity: 0.3; }}
            50% {{ opacity: 0.6; }}
        }}

        /* Section Titles with Rings */
        .section-title-with-ring {{
            display: flex;
            align-items: center;
            gap: 15px;
            font-size: 2em;
            font-weight: 300;
            letter-spacing: 3px;
            margin-bottom: 30px;
            padding-bottom: 15px;
            border-bottom: 2px solid rgba(102, 126, 234, 0.3);
            text-transform: uppercase;
            font-family: 'Courier New', 'Monaco', 'Menlo', monospace;
        }}

        .section-title-text {{
            background: linear-gradient(45deg, #667eea, #764ba2);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
        }}

        .section-title {{
            font-size: 2em;
            font-weight: 300;
            letter-spacing: 3px;
            margin-bottom: 30px;
            padding-bottom: 15px;
            border-bottom: 2px solid rgba(102, 126, 234, 0.3);
            text-transform: uppercase;
            font-family: 'Courier New', 'Monaco', 'Menlo', monospace;
        }}

        .color-legend {{
            background: rgba(255, 255, 255, 0.03);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 20px;
            padding: 30px;
            margin-bottom: 40px;
        }}

        .legend-title {{
            font-size: 1.5em;
            font-weight: 400;
            margin-bottom: 10px;
            color: #667eea;
            text-align: center;
        }}

        .legend-subtitle {{
            font-size: 1em;
            color: #a0aec0;
            text-align: center;
            margin-bottom: 25px;
        }}

        .legend-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 20px;
        }}

        .legend-item {{
            display: flex;
            align-items: center;
            gap: 15px;
            padding: 15px;
            background: rgba(0, 0, 0, 0.2);
            border-radius: 10px;
        }}

        .legend-color {{
            width: 30px;
            height: 30px;
            border-radius: 50%;
            flex-shrink: 0;
        }}

        .legend-color.long-term-up {{
            background: linear-gradient(135deg, #3498db, #5dade2);
        }}

        .legend-color.long-term-down {{
            background: linear-gradient(135deg, #e67e22, #f39c12);
        }}

        .legend-color.short-term-up {{
            background: linear-gradient(135deg, #27ae60, #2ecc71);
        }}

        .legend-color.short-term-down {{
            background: linear-gradient(135deg, #e74c3c, #c0392b);
        }}

        .legend-text {{
            font-weight: 500;
            font-size: 1.1em;
        }}

        .chain-card {{
            background: rgba(255, 255, 255, 0.03);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 20px;
            padding: 30px;
            margin-bottom: 30px;
            transition: all 0.3s ease;
        }}

        .chain-header {{
            display: flex;
            justify-content: space-between;
            align-items: flex-start;
            margin-bottom: 20px;
        }}

        .chain-number {{
            background: linear-gradient(135deg, #667eea, #764ba2);
            width: 40px;
            height: 40px;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: 600;
            font-size: 1.2em;
            margin-right: 15px;
            flex-shrink: 0;
        }}

        .chain-title {{
            font-size: 1.1em;
            font-weight: 400;
            color: #ffffff;
            line-height: 1.4;
        }}

        .direction-badge {{
            padding: 8px 20px;
            border-radius: 30px;
            font-weight: 500;
            font-size: 0.85em;
            text-transform: uppercase;
            letter-spacing: 1px;
            white-space: nowrap;
        }}

        .direction-badge.short-term-up {{
            background: rgba(39, 174, 96, 0.2);
            color: #27ae60;
            border: 1px solid #27ae60;
        }}

        .direction-badge.short-term-down {{
            background: rgba(231, 76, 60, 0.2);
            color: #e74c3c;
            border: 1px solid #e74c3c;
        }}

        .direction-badge.long-term-up {{
            background: rgba(52, 152, 219, 0.2);
            color: #3498db;
            border: 1px solid #3498db;
        }}

        .direction-badge.long-term-down {{
            background: rgba(230, 126, 34, 0.2);
            color: #e67e22;
            border: 1px solid #e67e22;
        }}

        .chain-content {{
            background: rgba(0, 0, 0, 0.2);
            border-radius: 10px;
            padding: 20px;
            margin-bottom: 20px;
            font-family: 'Courier New', monospace;
            font-size: 0.95em;
            line-height: 1.6;
            color: #cbd5e0;
        }}

        .mermaid-wrapper {{
            background: rgba(255, 255, 255, 0.05);
            border-radius: 10px;
            padding: 20px;
            overflow-x: auto;
        }}

        .summary-card {{
            background: rgba(255, 255, 255, 0.03);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 20px;
            padding: 40px;
            margin-bottom: 30px;
        }}

        .summary-content {{
            font-size: 1.1em;
            line-height: 1.8;
            color: #cbd5e0;
        }}

        .summary-content p {{
            margin-bottom: 15px;
        }}

        /* TWO COLUMN LAYOUT FOR SELL/BUY */
        .sell-buy-container {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 30px;
            margin-bottom: 30px;
        }}

        .sell-side-card {{
            background: rgba(231, 76, 60, 0.05);
            border: 1px solid rgba(231, 76, 60, 0.2);
            border-radius: 20px;
            padding: 30px;
        }}

        .buy-side-card {{
            background: rgba(39, 174, 96, 0.05);
            border: 1px solid rgba(39, 174, 96, 0.2);
            border-radius: 20px;
            padding: 30px;
        }}

        .side-title {{
            font-size: 1.3em;
            font-weight: 600;
            margin-bottom: 20px;
            text-align: center;
            text-transform: uppercase;
            letter-spacing: 2px;
        }}

        .sell-side-title {{
            color: #e74c3c;
        }}

        .buy-side-title {{
            color: #27ae60;
        }}

        .side-content {{
            font-size: 1em;
            line-height: 1.7;
            color: #cbd5e0;
        }}

        .side-content p {{
            margin-bottom: 12px;
        }}

        /* NEW: FOUR PANEL LAYOUT FOR RISK/REWARD */
        .risk-reward-grid {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 30px;
            margin-bottom: 40px;
        }}

        .risk-panel {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
        }}

        .reward-panel {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
        }}

        .metric-card {{
            background: rgba(255, 255, 255, 0.03);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 20px;
            padding: 30px;
            text-align: center;
            transition: all 0.3s ease;
        }}

        .metric-value {{
            font-size: 3em;
            font-weight: 600;
            margin-bottom: 10px;
        }}

        /* NEW COLOR SCHEME: Higher scores = Green, Lower scores = Red */
        .metric-value.excellent {{
            color: #27ae60;
            text-shadow: 0 0 20px rgba(39, 174, 96, 0.5);
        }}

        .metric-value.good {{
            color: #2ecc71;
            text-shadow: 0 0 20px rgba(46, 204, 113, 0.5);
        }}

        .metric-value.warning {{
            color: #f39c12;
            text-shadow: 0 0 20px rgba(243, 156, 18, 0.5);
        }}

        .metric-value.danger {{
            color: #e74c3c;
            text-shadow: 0 0 20px rgba(231, 76, 60, 0.5);
        }}

        .metric-label {{
            font-size: 0.9em;
            color: #a0aec0;
            text-transform: uppercase;
            letter-spacing: 1px;
        }}

        /* NEW: Dynamic Reasoning Bar */
        .reasoning-container {{
            background: rgba(102, 126, 234, 0.1);
            border: 1px solid rgba(102, 126, 234, 0.3);
            border-radius: 20px;
            padding: 30px;
            margin-bottom: 40px;
        }}

        .reasoning-title {{
            font-size: 1.3em;
            margin-bottom: 20px;
            color: #667eea;
            font-weight: 500;
            text-align: center;
        }}

        .reasoning-bar {{
            background: rgba(0, 0, 0, 0.3);
            border-radius: 15px;
            padding: 25px;
            border-left: 4px solid #667eea;
        }}

        .reasoning-text {{
            font-size: 1.1em;
            line-height: 1.7;
            color: #cbd5e0;
            text-align: justify;
        }}

        .conclusion-card {{
            background: rgba(102, 126, 234, 0.1);
            border: 1px solid rgba(102, 126, 234, 0.3);
            border-radius: 20px;
            padding: 30px;
            margin-bottom: 20px;
        }}

        .conclusion-title {{
            font-size: 1.3em;
            margin-bottom: 15px;
            color: #667eea;
            font-weight: 500;
        }}

        .conclusion-text {{
            line-height: 1.7;
            color: #cbd5e0;
        }}

        .footer {{
            text-align: center;
            padding: 40px 20px;
            margin-top: 80px;
            border-top: 1px solid rgba(255, 255, 255, 0.1);
            color: #a0aec0;
            font-size: 0.9em;
        }}

        /* Responsive design */
        @media (max-width: 768px) {{
            .sell-buy-container {{
                grid-template-columns: 1fr;
            }}
            .risk-reward-grid {{
                grid-template-columns: 1fr;
            }}
            .risk-panel, .reward-panel {{
                grid-template-columns: 1fr;
            }}
        }}
    </style>
</head>
<body>
    <div class="main-container">
        <!-- Large Logo Section - Invisible Background -->
        <div class="logo-section">
            <svg class="logo-svg" viewBox="0 0 360 150" xmlns="http://www.w3.org/2000/svg">
                <defs>
                    <linearGradient id="cleanGradient" x1="0%" y1="0%" x2="100%" y2="100%">
                        <stop offset="0%" style="stop-color:#667eea;stop-opacity:1">
                            <animate attributeName="stop-color" 
                                    values="#667eea;#764ba2;#667eea" 
                                    dur="4s" repeatCount="indefinite"/>
                        </stop>
                        <stop offset="100%" style="stop-color:#764ba2;stop-opacity:1">
                            <animate attributeName="stop-color" 
                                    values="#764ba2;#667eea;#764ba2" 
                                    dur="4s" repeatCount="indefinite"/>
                        </stop>
                    </linearGradient>
                    <filter id="subtleGlow" x="-20%" y="-20%" width="140%" height="140%">
                        <feGaussianBlur stdDeviation="3" result="coloredBlur"/>
                        <feMerge> 
                            <feMergeNode in="coloredBlur"/>
                            <feMergeNode in="SourceGraphic"/>
                        </feMerge>
                    </filter>
                </defs>
                
                <!-- First Ellipse (Quantitative) -->
                <ellipse cx="150" cy="65" rx="50" ry="25" 
                      fill="none"
                      stroke="url(#cleanGradient)" 
                      stroke-width="5"
                      filter="url(#subtleGlow)"
                      opacity="0.9">
                    <animate attributeName="opacity" values="0.9;1;0.9" dur="3s" repeatCount="indefinite"/>
                </ellipse>

                <!-- Second Ellipse (Qualitative) -->
                <ellipse cx="210" cy="85" rx="50" ry="25" 
                      fill="none"
                      stroke="url(#cleanGradient)" 
                      stroke-width="5"
                      filter="url(#subtleGlow)"
                      opacity="0.9">
                    <animate attributeName="opacity" values="0.9;1;0.9" dur="3s" repeatCount="indefinite" begin="1.5s"/>
                </ellipse>

                <!-- Intersection area highlight -->
                <ellipse cx="180" cy="75" rx="20" ry="12" 
                      fill="url(#cleanGradient)" 
                      opacity="0.25"
                      filter="url(#subtleGlow)">
                    <animate attributeName="opacity" values="0.25;0.4;0.25" dur="3s" repeatCount="indefinite"/>
                </ellipse>
            </svg>
            <div class="logo-text">Q&Q.AI</div>
            <div class="logo-description">{logo_description}</div>
        </div>

        <!-- Ticker info bar -->
        <div class="ticker-bar">
            <div class="ticker-symbol">{ticker_symbol}</div>
            <div>Analysis Date: {analysis_date}</div>
            <div>{chain_count} Impact Chains Analyzed</div>
        </div>

        <!-- Color Legend -->
        {color_legend_html}

        <!-- Impact Chains Section - NEW TITLE WITH RING -->
        <div class="section">
            {impact_chains_title_html}
            {chain_cards_html}
        </div>

        <!-- Sell & Buy Analysis - TWO COLUMN LAYOUT -->
        <div class="section">
            <h2 class="section-title">{sell_buy_title}</h2>
            <div class="sell-buy-container">
                <div class="sell-side-card">
                    <h3 class="side-title sell-side-title">{sell_side_title}</h3>
                    <div class="side-content">
                        {sell_side_html}
                    </div>
                </div>
                <div class="buy-side-card">
                    <h3 class="side-title buy-side-title">{buy_side_title}</h3>
                    <div class="side-content">
                        {buy_side_html}
                    </div>
                </div>
            </div>
        </div>

        <!-- Business Logic Summary -->
        <div class="section">
            <h2 class="section-title">{business_logic_title}</h2>
            <div class="summary-card">
                <div class="summary-content">
                    <p>{business_logic_data}</p>
                </div>
            </div>
        </div>

        <!-- Conclusion Analysis - NEW TITLE WITH RING -->
        <div class="section">
            {conclusion_title_html}

            <!-- NEW: Four Panel Risk/Reward Layout -->
            <div class="risk-reward-grid">
                <!-- Risk Panel -->
                <div class="risk-panel">
                    <div class="metric-card">
                        <div class="metric-value {short_term_risk_class}">{short_term_risk}/1.0</div>
                        <div class="metric-label">{short_term_risk_text}</div>
                    </div>
                    <div class="metric-card">
                        <div class="metric-value {long_term_risk_class}">{long_term_risk}/1.0</div>
                        <div class="metric-label">{long_term_risk_text}</div>
                    </div>
                </div>
                
                <!-- Reward Panel -->
                <div class="reward-panel">
                    <div class="metric-card">
                        <div class="metric-value {short_term_reward_class}">{short_term_reward}/1.0</div>
                        <div class="metric-label">{short_term_reward_text}</div>
                    </div>
                    <div class="metric-card">
                        <div class="metric-value {long_term_reward_class}">{long_term_reward}/1.0</div>
                        <div class="metric-label">{long_term_reward_text}</div>
                    </div>
                </div>
            </div>

            <!-- NEW: Dynamic Reasoning Bar -->
            <div class="reasoning-container">
                <h3 class="reasoning-title">{dynamic_reasoning_title}</h3>
                <div class="reasoning-bar">
                    <div class="reasoning-text">{dynamic_reasoning}</div>
                </div>
            </div>

            <!-- Dynamic Rating -->
            <div class="conclusion-card">
                <h3 class="conclusion-title">{dynamic_rating_text}</h3>
                <div class="conclusion-text">{dynamic_rating}</div>
            </div>

            <!-- Key Catalysts -->
            <div class="conclusion-card">
                <h3 class="conclusion-title">{key_catalysts_text}</h3>
                <div class="conclusion-text">{catalyst}</div>
            </div>

            <!-- Short Term Impact -->
            <div class="conclusion-card">
                <h3 class="conclusion-title">{short_term_impact_text}</h3>
                <div class="conclusion-text">{short_term_impact}</div>
            </div>

            <!-- Long Term Outlook -->
            <div class="conclusion-card">
                <h3 class="conclusion-title">{long_term_outlook_text}</h3>
                <div class="conclusion-text">{long_term_outlook}</div>
            </div>
        </div>

        <!-- Footer -->
        <div class="footer">
            <p>{footer_text}</p>
            <p>{report_generated_text}: {report_date_time}</p>
            <p>{copyright_text}</p>
        </div>
    </div>

    <script>
        // Initialize Mermaid
        mermaid.initialize({{
            startOnLoad: true,
            theme: 'dark',
            themeVariables: {{
                primaryColor: '#667eea',
                primaryTextColor: '#fff',
                primaryBorderColor: '#764ba2',
                lineColor: '#667eea',
                secondaryColor: '#764ba2',
                background: 'rgba(255, 255, 255, 0.05)',
                mainBkg: 'rgba(255, 255, 255, 0.05)',
                secondBkg: 'rgba(102, 126, 234, 0.1)',
                fontFamily: 'Segoe UI',
                fontSize: '14px'
            }},
            flowchart: {{
                useMaxWidth: true,
                htmlLabels: true,
                curve: 'basis'
            }}
        }});
    </script>
</body>
</html>"""
    
    # Save to file
    filename = f"qq_ai_report_{ticker_symbol}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    # Open in browser
    webbrowser.open(f"file://{os.path.abspath(filename)}")
    
    print(f"✅ Q&Q.AI Report generated: {filename}")
    print(f"�� Total chains: {len(chain_data)}")
    print(f"⚠️ Short Term Risk: {short_term_risk}/1.0 ({short_term_risk_class})")
    print(f"⚠️ Long Term Risk: {long_term_risk}/1.0 ({long_term_risk_class})")
    print(f"🚀 Short Term Reward: {short_term_reward:.2f}/1.0 ({short_term_reward_class})")
    print(f"🚀 Long Term Reward: {long_term_reward:.2f}/1.0 ({long_term_reward_class})")
    print(f"📊 Sell side items: {len(sell_side_data)}")
    print(f"🔍 Buy side items: {len(buy_side_data)}")
    print(f"🏢 Business logic extracted: {'Yes' if business_logic_data != 'No business logic available' else 'No'}")
    print(f"🌐 Language: {'Chinese' if is_chinese else 'English'}")
    
    return filename

# Run it - Large logo design!
visualize_qq_ai_report()

✅ Successfully extracted from internet_search_result:
   - Sell side items: 6
   - Buy side items: 6
   - Business logic items: 6
✅ Dynamic Rating scores extracted:
   - Short Term Risk: 0.75
   - Short Term Reward: 0.25
   - Long Term Risk: 0.35
   - Long Term Reward: 0.85
✅ Q&Q.AI Report generated: qq_ai_report_PLTR_20250909_111532.html
�� Total chains: 6
⚠️ Short Term Risk: 0.75/1.0 (danger)
⚠️ Long Term Risk: 0.35/1.0 (good)
🚀 Short Term Reward: 0.25/1.0 (danger)
🚀 Long Term Reward: 0.85/1.0 (excellent)
📊 Sell side items: 6
🔍 Buy side items: 6
🏢 Business logic extracted: Yes
🌐 Language: Chinese


'qq_ai_report_PLTR_20250909_111532.html'

### Frontend Print This Code

# --------------------------------------- Above is Supervisor to Frontend Logic --------------------------------

### Update Logic For the whole Pipeline 

In [263]:
from tavily import TavilyClient

In [264]:

def update_check(ticker):
    """
    Check the most updated news for a ticker with detailed bullet point summary
    
    Args:
        ticker (str): Stock ticker symbol (e.g., 'MSTR', 'NVDA')
    
    Returns:
        str: Summarized sources with bullet points
    """
    try:
        # Initialize Tavily client
        client = TavilyClient("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")
        
        print(f"🔍 Checking latest updates for {ticker}...")
        
        # Search for most recent news and updates
        query = f"Latest news and updates for {ticker} stock - recent developments, earnings, announcements, analyst ratings, price movements. Provide detailed bullet point summary of key information."
        
        response = client.search(
            query=query,
            include_answer="advanced",
            search_depth="advanced",
            topic="finance",
            max_results=15
        )
        
        # Extract content from results
        content_list = []
        if 'results' in response:
            for result in response['results']:
                if 'content' in result and result['content']:
                    content_list.append(result['content'])
        
        # Get the answer summary
        answer = response.get('answer', '')
        
        # Create summarized sources
        summarized_sources = f"""
📰 LATEST UPDATES FOR {ticker.upper()}:
{answer}

📊 DETAILED SOURCES:
"""
        
        # Add bullet points from each source
        for i, content in enumerate(content_list[:10]):  # Limit to top 10 sources
            summarized_sources += f"• Source {i+1}: {content[:200]}...\n"
        
        print(f"✅ Update check complete for {ticker}")
        print(f"📈 Sources found: {len(response.get('results', []))}")
        print(f"⏱️ Response time: {response.get('response_time', 0)} seconds")
        
        return summarized_sources
        
    except Exception as e:
        print(f"❌ Error in update check for {ticker}: {e}")
        return f"❌ Error occurred during update check for {ticker}: {e}"

# Usage example:
# latest_updates = update_check("MSTR")
# print(latest_updates)